# Wheat Classification (placeholder)

# Phase 4 (Sklearn) — Wheat Seeds Classification

This notebook trains and evaluates multiple classifiers on the **Seeds Dataset** (`seeds_dataset.txt`).

**Goal:** predict the wheat variety class (3 classes) using 7 numeric features.

In [ ]:
# Imports and basic configuration
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier

import matplotlib.pyplot as plt
import seaborn as sns

RANDOM_STATE = 42
pd.set_option('display.max_columns', 50)
sns.set_theme(style='whitegrid')

## 1) Load dataset
The dataset file is expected at `./seeds_dataset.txt` (same folder as this notebook).

In [ ]:
# Load the seeds dataset (whitespace-separated, no header)
data_path = 'seeds_dataset.txt'

columns = [
    'area',
    'perimeter',
    'compactness',
    'kernel_length',
    'kernel_width',
    'asymmetry_coefficient',
    'kernel_groove_length',
    'class'
]

df = pd.read_csv(data_path, sep=r'\s+', header=None, names=columns)
df.head()

In [ ]:
# Quick validation
print('Shape:', df.shape)
print('Missing values:', int(df.isna().sum().sum()))
print('Class distribution:')
display(df['class'].value_counts().sort_index())

df.describe()

## 2) Exploratory analysis

In [ ]:
# Correlation heatmap
plt.figure(figsize=(10, 7))
corr = df.drop(columns=['class']).corr()
sns.heatmap(corr, annot=False, cmap='viridis', square=True)
plt.title('Feature Correlation (Seeds Dataset)')
plt.tight_layout()
plt.show()

In [ ]:
# Pairplot (can be a bit heavy, but useful for small datasets)
sns.pairplot(df, hue='class', corner=True, diag_kind='hist')
plt.show()

## 3) Train/test split
We keep a hold-out test set and use cross-validation on the training set.

In [ ]:
X = df.drop(columns=['class'])
y = df['class'].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

print('Train:', X_train.shape, 'Test:', X_test.shape)

## 4) Baseline models (cross-validation)
We compare a few common classifiers. Models that are sensitive to feature scale use `StandardScaler`.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

candidates = {
    'LogisticRegression': Pipeline([
        ('scaler', StandardScaler()),
        ('model', LogisticRegression(max_iter=500, random_state=RANDOM_STATE))
    ]),
    'KNN': Pipeline([
        ('scaler', StandardScaler()),
        ('model', KNeighborsClassifier())
    ]),
    'SVM (RBF)': Pipeline([
        ('scaler', StandardScaler()),
        ('model', SVC(kernel='rbf', probability=False, random_state=RANDOM_STATE))
    ]),
    'GaussianNB': GaussianNB(),
    'RandomForest': RandomForestClassifier(n_estimators=300, random_state=RANDOM_STATE)
}

rows = []
for name, model in candidates.items():
    scores = cross_val_score(model, X_train, y_train, cv=cv, scoring='accuracy')
    rows.append({
        'model': name,
        'cv_accuracy_mean': float(scores.mean()),
        'cv_accuracy_std': float(scores.std())
    })

results = pd.DataFrame(rows).sort_values('cv_accuracy_mean', ascending=False)
results

## 5) Train the best baseline and evaluate on the test set

In [ ]:
best_name = results.iloc[0]['model']
best_baseline = candidates[best_name]

best_baseline.fit(X_train, y_train)
y_pred = best_baseline.predict(X_test)

print('Best baseline:', best_name)
print('Test accuracy:', accuracy_score(y_test, y_pred))
print('\nClassification report:')
print(classification_report(y_test, y_pred))

In [ ]:
# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title(f'Confusion Matrix — {best_name}')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.tight_layout()
plt.show()

## 6) Hyperparameter tuning (GridSearchCV)
We tune one or two strong candidates (KNN and SVM are typical for this dataset).

In [ ]:
# Grid search: SVM (RBF)
svm_pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('model', SVC(kernel='rbf', random_state=RANDOM_STATE))
])

svm_param_grid = {
    'model__C': [0.1, 1, 10, 100],
    'model__gamma': ['scale', 0.01, 0.1, 1]
}

svm_search = GridSearchCV(
    svm_pipe,
    param_grid=svm_param_grid,
    cv=cv,
    scoring='accuracy',
    n_jobs=-1
)

svm_search.fit(X_train, y_train)
print('Best CV accuracy (SVM):', svm_search.best_score_)
print('Best params:', svm_search.best_params_)

In [ ]:
# Grid search: KNN
knn_pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('model', KNeighborsClassifier())
])

knn_param_grid = {
    'model__n_neighbors': list(range(3, 22, 2)),
    'model__weights': ['uniform', 'distance'],
    'model__p': [1, 2]
}

knn_search = GridSearchCV(
    knn_pipe,
    param_grid=knn_param_grid,
    cv=cv,
    scoring='accuracy',
    n_jobs=-1
)

knn_search.fit(X_train, y_train)
print('Best CV accuracy (KNN):', knn_search.best_score_)
print('Best params:', knn_search.best_params_)

In [ ]:
# Evaluate the best tuned model on the test set
tuned_candidates = {
    'SVM (tuned)': svm_search.best_estimator_,
    'KNN (tuned)': knn_search.best_estimator_
}

tuned_rows = []
for name, model in tuned_candidates.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    tuned_rows.append({
        'model': name,
        'test_accuracy': float(accuracy_score(y_test, pred))
    })

pd.DataFrame(tuned_rows).sort_values('test_accuracy', ascending=False)

## 7) Conclusion
You now have a full Phase 4 Sklearn deliverable: dataset loading, EDA, model comparison, evaluation, and tuning.

# Fase 4 - Classificação de Grãos de Trigo (Seeds Dataset)

Este notebook implementa a atividade IR ALÉM com CRISP-DM: EDA, classificação, otimização e interpretação.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression

sns.set_theme(style='whitegrid', palette='viridis')

In [ ]:
columns = [
    'area','perimeter','compactness','kernel_length','kernel_width',
    'asymmetry_coefficient','kernel_groove_length','class'
]

df = pd.read_csv('seeds_dataset.txt', sep='\s+', header=None, names=columns)
df['class_name'] = df['class'].map({1: 'Kama', 2: 'Rosa', 3: 'Canadian'})

print('Shape:', df.shape)
display(df.head())
display(df.describe().T)
print('Valores ausentes:')
print(df.isnull().sum())

In [ ]:
num_cols = columns[:-1]

fig, axes = plt.subplots(3, 3, figsize=(16, 11))
axes = axes.flatten()
for i, col in enumerate(num_cols):
    sns.histplot(df[col], kde=True, ax=axes[i], color='teal')
    axes[i].set_title(f'Histograma - {col}')
axes[-1].axis('off')
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 7))
corr = df[num_cols].corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', square=True)
plt.title('Matriz de Correlação')
plt.tight_layout()
plt.show()

In [ ]:
X = df[num_cols].copy()
y = df['class'].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)

models = {
    'KNN': Pipeline([('scaler', StandardScaler()), ('model', KNeighborsClassifier())]),
    'SVM': Pipeline([('scaler', StandardScaler()), ('model', SVC())]),
    'RandomForest': Pipeline([('scaler', StandardScaler()), ('model', RandomForestClassifier(random_state=42))]),
    'NaiveBayes': Pipeline([('scaler', StandardScaler()), ('model', GaussianNB())]),
    'LogisticRegression': Pipeline([('scaler', StandardScaler()), ('model', LogisticRegression(max_iter=2000, random_state=42))])
}

rows = []
predictions = {}

for name, pipeline in models.items():
    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)
    predictions[name] = y_pred
    rows.append({
        'model': name,
        'accuracy': accuracy_score(y_test, y_pred),
        'precision_macro': precision_score(y_test, y_pred, average='macro'),
        'recall_macro': recall_score(y_test, y_pred, average='macro'),
        'f1_macro': f1_score(y_test, y_pred, average='macro')
    })

results_df = pd.DataFrame(rows).sort_values('f1_macro', ascending=False).reset_index(drop=True)
display(results_df)

In [ ]:
best_model_name = results_df.iloc[0]['model']
best_model_pred = predictions[best_model_name]
print(f'Melhor modelo inicial: {best_model_name}')
print(classification_report(y_test, best_model_pred, target_names=['Kama', 'Rosa', 'Canadian']))

cm = confusion_matrix(y_test, best_model_pred)
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Kama', 'Rosa', 'Canadian'], yticklabels=['Kama', 'Rosa', 'Canadian'])
plt.title(f'Matriz de Confusão - {best_model_name}')
plt.xlabel('Predito')
plt.ylabel('Real')
plt.tight_layout()
plt.show()

In [ ]:
param_grids = {
    'KNN': {'model__n_neighbors': [3, 5, 7, 9], 'model__weights': ['uniform', 'distance']},
    'SVM': {'model__C': [0.1, 1, 10, 50], 'model__gamma': ['scale', 0.01, 0.1], 'model__kernel': ['rbf', 'linear']},
    'RandomForest': {'model__n_estimators': [100, 200], 'model__max_depth': [None, 5, 10], 'model__min_samples_split': [2, 5]}
}

optimized_rows = []
for name in ['KNN', 'SVM', 'RandomForest']:
    search = GridSearchCV(models[name], param_grids[name], cv=5, scoring='f1_macro', n_jobs=-1)
    search.fit(X_train, y_train)
    y_pred = search.best_estimator_.predict(X_test)
    optimized_rows.append({
        'model': name,
        'best_params': str(search.best_params_),
        'accuracy': accuracy_score(y_test, y_pred),
        'precision_macro': precision_score(y_test, y_pred, average='macro'),
        'recall_macro': recall_score(y_test, y_pred, average='macro'),
        'f1_macro': f1_score(y_test, y_pred, average='macro')
    })

optimized_df = pd.DataFrame(optimized_rows).sort_values('f1_macro', ascending=False).reset_index(drop=True)
display(optimized_df)

## Insights finais

- O dataset apresenta boa separação entre classes de grão.
- Modelos não lineares (SVM e RandomForest) tendem a apresentar melhor desempenho.
- A otimização de hiperparâmetros melhora a estabilidade do F1 macro.
- Para produção, recomenda-se monitorar drift de dados e re-treinar periodicamente.

# Fase 4 - Classificação de Grãos de Trigo (Seeds Dataset)

Este notebook implementa a atividade **IR ALÉM – Da Terra ao Código** com a metodologia **CRISP-DM**.

## Objetivos
1. Analisar e pré-processar os dados
2. Treinar e comparar múltiplos algoritmos de classificação
3. Otimizar modelos com GridSearch
4. Interpretar resultados e extrair insights

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression

sns.set_theme(style='whitegrid', palette='viridis')

## 1) Carregamento e inspeção inicial

In [ ]:
columns = [
    'area',
    'perimeter',
    'compactness',
    'kernel_length',
    'kernel_width',
    'asymmetry_coefficient',
    'kernel_groove_length',
    'class'
]

df = pd.read_csv('seeds_dataset.txt', sep='\s+', header=None, names=columns)
df['class_name'] = df['class'].map({1: 'Kama', 2: 'Rosa', 3: 'Canadian'})

print('Shape:', df.shape)
df.head()

In [ ]:
print('Tipos:')
print(df.dtypes)

print('\nValores ausentes por coluna:')
print(df.isnull().sum())

print('\nEstatísticas descritivas:')
df.describe().T

## 2) Análise exploratória (EDA)

In [ ]:
num_cols = columns[:-1]

fig, axes = plt.subplots(3, 3, figsize=(16, 11))
axes = axes.flatten()

for i, col in enumerate(num_cols):
    sns.histplot(df[col], kde=True, ax=axes[i], color='teal')
    axes[i].set_title(f'Histograma - {col}')

axes[-1].axis('off')
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(16, 11))
axes = axes.flatten()

for i, col in enumerate(num_cols):
    sns.boxplot(data=df, x='class_name', y=col, ax=axes[i])
    axes[i].set_title(f'Boxplot - {col}')

axes[-1].axis('off')
plt.tight_layout()
plt.show()

In [ ]:
sns.pairplot(df[['area', 'perimeter', 'kernel_length', 'kernel_width', 'class_name']], hue='class_name', corner=True)
plt.show()

In [ ]:
plt.figure(figsize=(10, 7))
corr = df[num_cols].corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', square=True)
plt.title('Matriz de Correlação')
plt.tight_layout()
plt.show()

## 3) Pré-processamento e divisão treino/teste

In [ ]:
X = df[num_cols].copy()
y = df['class'].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)

print('Treino:', X_train.shape, '| Teste:', X_test.shape)

## 4) Treinamento e comparação de algoritmos

In [ ]:
models = {
    'KNN': Pipeline([('scaler', StandardScaler()), ('model', KNeighborsClassifier())]),
    'SVM': Pipeline([('scaler', StandardScaler()), ('model', SVC())]),
    'RandomForest': Pipeline([('scaler', StandardScaler()), ('model', RandomForestClassifier(random_state=42))]),
    'NaiveBayes': Pipeline([('scaler', StandardScaler()), ('model', GaussianNB())]),
    'LogisticRegression': Pipeline([('scaler', StandardScaler()), ('model', LogisticRegression(max_iter=2000, random_state=42))])
}

rows = []
predictions = {}

for name, pipeline in models.items():
    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)
    predictions[name] = y_pred

    rows.append({
        'model': name,
        'accuracy': accuracy_score(y_test, y_pred),
        'precision_macro': precision_score(y_test, y_pred, average='macro'),
        'recall_macro': recall_score(y_test, y_pred, average='macro'),
        'f1_macro': f1_score(y_test, y_pred, average='macro')
    })

results_df = pd.DataFrame(rows).sort_values('f1_macro', ascending=False).reset_index(drop=True)
results_df

In [ ]:
plt.figure(figsize=(10, 5))
sns.barplot(data=results_df, x='model', y='f1_macro')
plt.title('Comparação de Modelos - F1 Macro')
plt.ylim(0.7, 1.01)
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

In [ ]:
best_model_name = results_df.iloc[0]['model']
best_model_pred = predictions[best_model_name]

print(f'Melhor modelo inicial: {best_model_name}')
print('\nClassification Report:')
print(classification_report(y_test, best_model_pred, target_names=['Kama', 'Rosa', 'Canadian']))

cm = confusion_matrix(y_test, best_model_pred)
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Kama', 'Rosa', 'Canadian'], yticklabels=['Kama', 'Rosa', 'Canadian'])
plt.title(f'Matriz de Confusão - {best_model_name}')
plt.xlabel('Predito')
plt.ylabel('Real')
plt.tight_layout()
plt.show()

## 5) Otimização de hiperparâmetros (Grid Search)

In [ ]:
param_grids = {
    'KNN': {
        'model__n_neighbors': [3, 5, 7, 9],
        'model__weights': ['uniform', 'distance']
    },
    'SVM': {
        'model__C': [0.1, 1, 10, 50],
        'model__gamma': ['scale', 0.01, 0.1, 1],
        'model__kernel': ['rbf', 'linear']
    },
    'RandomForest': {
        'model__n_estimators': [100, 200, 300],
        'model__max_depth': [None, 5, 10, 20],
        'model__min_samples_split': [2, 5, 10]
    }
}

optimized_rows = []

for name in ['KNN', 'SVM', 'RandomForest']:
    search = GridSearchCV(
        estimator=models[name],
        param_grid=param_grids[name],
        cv=5,
        scoring='f1_macro',
        n_jobs=-1
    )

    search.fit(X_train, y_train)
    y_pred = search.best_estimator_.predict(X_test)

    optimized_rows.append({
        'model': name,
        'best_params': str(search.best_params_),
        'accuracy': accuracy_score(y_test, y_pred),
        'precision_macro': precision_score(y_test, y_pred, average='macro'),
        'recall_macro': recall_score(y_test, y_pred, average='macro'),
        'f1_macro': f1_score(y_test, y_pred, average='macro')
    })

optimized_df = pd.DataFrame(optimized_rows).sort_values('f1_macro', ascending=False).reset_index(drop=True)
optimized_df

In [ ]:
baseline_best = results_df[results_df['model'].isin(['KNN', 'SVM', 'RandomForest'])][['model', 'f1_macro']].rename(columns={'f1_macro': 'f1_baseline'})
comparison = optimized_df[['model', 'f1_macro']].rename(columns={'f1_macro': 'f1_optimized'}).merge(baseline_best, on='model')
comparison['gain'] = comparison['f1_optimized'] - comparison['f1_baseline']
comparison.sort_values('gain', ascending=False)

## 6) Interpretação e Insights

- O dataset possui boa separação entre classes, o que favorece alta performance.
- Modelos não-lineares (SVM/RandomForest) tendem a capturar melhor interações entre atributos físicos dos grãos.
- A otimização de hiperparâmetros pode trazer ganhos de desempenho, principalmente em F1 macro.
- Para uso real em cooperativas, recomenda-se monitoramento contínuo de desempenho e re-treino periódico com novos dados.